In [ ]:
#!pip install soundfile

In [ ]:
import importlib
from dataclasses import dataclass
from typing import Any, Dict, Optional
import numpy as np
import math
from tqdm import tqdm
import io
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from itertools import islice

import library.utils.general
importlib.reload(library.utils.general)
from library.utils.general import *
import library.utils.tokenization
importlib.reload(library.utils.tokenization)
from library.utils.tokenization import *

import library.model.audio
importlib.reload(library.model.audio)
from library.model.audio import *

In [ ]:
try:
    from datasets import load_dataset, Audio as HFAudio
    _HAS_HF = True
except Exception:
    _HAS_HF = False
try:
    import soundfile as sf
    _HAS_SF = True
except Exception:
    sf = None
    _HAS_SF = False
try:
    import audioread
    _HAS_AR = True
except Exception:
    audioread = None
    _HAS_AR = False
try:
    import torchaudio
    _HAS_TA = True
except Exception:
    torchaudio = None
    _HAS_TA = False

In [ ]:
DEFAULT_TEXT_LENGTH_TOKENS = 32
DEFAULT_AUDIO_LENGTH_SECONDS = 8
SAMPLE_RATE = 16000

vocab = TextVocabulary()

vocab.add_tokens([str(t).upper() for t in [chr(i) for i in range(32, 127)]])
vocab.add_tokens([str(t).upper() for t in [
    "un", "re", "in", "im", "il", "ir", "dis", "en", "em", "non",
    "in", "im", "over", "mis", "sub", "pre", "inter", "fore", "de", "trans",
    "super", "semi", "anti", "mid", "under", "hyper", "auto", "bio", "micro",
    "macro", "mini", "mono", "multi", "poly", "tri", "uni", "a", "ab", "ad",
    "ambi", "amphi", "ante", "anti", "arch", "be", "bi", "circum", "co", "col",
    "com", "con", "contra", "counter", "de", "dia", "epi", "ex", "extra",
    "fore", "hetero", "homo", "hyper", "hypo", "intra", "inter", "intro",
    "mal", "mis", "ob", "omni", "para", "peri", "post", "pre", "pro", "pseudo",
    "retro", "semi", "sub", "super", "syn", "sym", "tele", "trans", "ultra", "vice"
]])
vocab.add_tokens([str(t).upper() for t in [
    "able", "ible", "al", "ial", "ed", "en", "er", "est", "ful", "ic", "ing",
    "ion", "tion", "ation", "ition", "ment", "ness", "ous", "eous", "ious",
    "less", "ly", "ise", "ize", "fy", "ish", "ism", "ist", "ity", "ty", "ment",
    "ness", "ship", "sion", "tion", "acy", "al", "ance", "ence", "dom", "er",
    "or", "ess", "est", "hood", "ian", "ate", "en", "ful", "ive", "ward", "wise",
    "y", "ed", "ing", "es", "est", "ese", "ette", "let", "ling", "logy", "ology",
    "most", "less", "ous", "eous", "ious"
]])

#vocab.add_tokens(VOCAB_ASCII)
#vocab.add_tokens(VOCAB_INTEGERS)
PAD_ID = vocab.PAD_ID
EOS_ID = vocab.EOS_ID
SEP_ID = vocab.token_to_id("<|SEP|>")
VOCAB_SIZE = vocab.vocab_size

device = get_device()
vocab.tokens

In [ ]:
specs = [
    DatasetSpec(source="seastar105/emo_speech_caption_test", split="train", field_map={"text": "transcript", "audio": "audio"}),
    DatasetSpec(source="ylacombe/podcast_fillers_processed", split="train", field_map={"text": "text", "audio": "audio"}),
    DatasetSpec(source="ahazeemi/librispeech10h", split="train.10", field_map={"text": "text", "audio": "audio"}),
]

ds = MultiModalAudioText(specs)

print("_HAS_TA:", _HAS_TA)
print("torchaudio:", torchaudio.__version__ if _HAS_TA else None)

di, (ds0, fm0) = 0, ds.datasets[0]
row0 = ds0[0]
raw = get(row0, fm0.get("audio"))

import io
audio = to_audio_wave(raw)
print("decoded?", audio is not None, "sr:", None if audio is None else audio[1],
      "shape:", None if audio is None else tuple(audio[0].shape))

collate_fn = make_collate_audio_text(vocab, max_text_len=DEFAULT_TEXT_LENGTH_TOKENS, audio_sr=SAMPLE_RATE, audio_len=DEFAULT_AUDIO_LENGTH_SECONDS)
loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0, collate_fn=collate_fn, drop_last=True)

print("DEVICE:", device, "| len(ds):", len(ds))

In [ ]:
#codec = AudioCodec(D=64, hop=256, K=16, n_codes=128, beta=0.25, drop=0.1)
codec = AudioCodec(D=256, hop=128, K=16, n_codes=128, beta=0.25, drop=0.1).to(device)
#codec.load_state_dict(torch.load("audio.codec.pt", map_location="cpu"))# save 64, 128, 16, 512
load_partial_state_dict(codec,f"../artifact/audio.codec.lg.pt", map_location=torch.device('cpu'))

codec = codec.to(device)

print_model_params_count(codec)

In [ ]:
import math

def print_codec_compression(codec, sr=SAMPLE_RATE, pcm_bits=16):
    hop = codec.hop
    K = codec.K
    n_codes = codec.n_codes

    bits_per_index = math.log2(n_codes)
    tokens_per_sec = sr / hop
    bits_per_token = K * bits_per_index

    codec_bps = tokens_per_sec * bits_per_token
    codec_kbps = codec_bps / 1000

    raw_bps = sr * pcm_bits
    raw_kbps = raw_bps / 1000

    compression_ratio = raw_bps / codec_bps

    print("=== Audio Codec Compression ===")
    print(f"sample rate:        {sr} Hz")
    print(f"hop:                {hop} samples")
    print(f"tokens/sec:         {tokens_per_sec:.2f}")

    print()
    print("=== Codebooks ===")
    print(f"RVQ stages (K):     {K}")
    print(f"codes per book:     {n_codes}")
    print(f"bits per index:     {bits_per_index:.2f}")

    print()
    print("=== Bitrate ===")
    print(f"bits per token:     {bits_per_token:.2f}")
    print(f"codec bitrate:      {codec_kbps:.2f} kbps")

    print()
    print("=== Raw Audio ===")
    print(f"PCM bitrate:        {raw_kbps:.2f} kbps  ({pcm_bits}-bit mono)")

    print()
    print("=== Compression ===")
    print(f"compression ratio:  {compression_ratio:.2f}x smaller")

# run it
print_codec_compression(codec)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.filterwarnings(
    "ignore",
    message="An output with one or more elements was resized",
    category=UserWarning,
)

# If you want to silence ALL torch stft resize warnings more aggressively:
import torch
torch._C._set_warnAlways(False)

opt = torch.optim.AdamW(codec.parameters(), lr=1e-5, betas=(0.9, 0.95), weight_decay=0.01)

@torch.no_grad()
def usage_metrics(codes, n_codes: int):
    B, K, T = codes.shape
    ents = []
    effs = []
    for k in range(K):
        flat = codes[:, k, :].reshape(-1)
        hist = torch.bincount(flat, minlength=n_codes).float()
        p = hist / hist.sum().clamp_min(1.0)
        H = -(p * (p + 1e-9).log()).sum()
        ents.append(H)
        effs.append(torch.exp(H))
    Hm = torch.stack(ents).mean()
    Em = torch.stack(effs).mean()
    return float(Hm), float(Em)

_STFT_WIN = {}

def _stft_mag_unfold(x, n_fft=512, hop=128):
    B, T = x.shape
    device = x.device
    dtype = x.dtype

    key = (n_fft, device, dtype)
    win = _STFT_WIN.get(key)
    if win is None:
        win = torch.hann_window(n_fft, device=device, dtype=dtype)
        _STFT_WIN[key] = win

    if T < n_fft:
        Freq = n_fft // 2 + 1
        return x.new_zeros((B, Freq, 0))

    frames = 1 + (T - n_fft) // hop
    xw = x.unfold(dimension=1, size=n_fft, step=hop)  # (B, frames, n_fft)
    xw = xw * win.view(1, 1, n_fft)
    X = torch.fft.rfft(xw, n=n_fft, dim=-1)           # (B, frames, Freq)
    return X.abs().transpose(1, 2).contiguous()       # (B, Freq, frames)

def stft_mag_loss(x, y, n_fft=512, hop=128):
    X = _stft_mag_unfold(x, n_fft=n_fft, hop=hop)
    Y = _stft_mag_unfold(y, n_fft=n_fft, hop=hop)
    if X.numel() == 0:
        return x.new_tensor(0.0)
    return (X - Y).abs().mean()

def hf_diff_loss(x, y):
    dx = x[:, 1:] - x[:, :-1]
    dy = y[:, 1:] - y[:, :-1]
    return (dx - dy).abs().mean()

def soft_usage_loss_from_dist(dist, temp=0.5):
    B, K, T, N = dist.shape
    p = torch.softmax(-dist / temp, dim=-1)
    p_bar = p.mean(dim=(0, 2))
    ent = -(p_bar * (p_bar + 1e-9).log()).sum(dim=-1)
    Hnorm = ent / math.log(N)
    return (1.0 - Hnorm).mean()

def wav_mask_from_wav_len(wav_len: torch.Tensor, T: int, device):
    wav_len = wav_len.to(device=device)
    ar = torch.arange(T, device=device).unsqueeze(0)
    return ar < wav_len.unsqueeze(1)

def masked_l1(x, y, mask_bool: torch.Tensor):
    diff = (x - y).abs() * mask_bool.to(dtype=x.dtype)
    denom = mask_bool.sum().clamp_min(1).to(dtype=x.dtype)
    return diff.sum() / denom

def masked_tv_diff(wav_hat, wav, wav_mask):
    dv_hat = wav_hat[:, 1:] - wav_hat[:, :-1]
    dv = wav[:, 1:] - wav[:, :-1]
    m = wav_mask[:, 1:] & wav_mask[:, :-1]
    diff = (dv_hat - dv).abs() * m.to(dtype=dv_hat.dtype)
    denom = m.sum().clamp_min(1).to(dtype=dv_hat.dtype)
    return diff.sum() / denom

def masked_hf_diff_loss(wav_hat, wav, wav_mask):
    dx = wav_hat[:, 1:] - wav_hat[:, :-1]
    dy = wav[:, 1:] - wav[:, :-1]
    m = wav_mask[:, 1:] & wav_mask[:, :-1]
    diff = (dx - dy).abs() * m.to(dtype=dx.dtype)
    denom = m.sum().clamp_min(1).to(dtype=dx.dtype)
    return diff.sum() / denom

def stft_mag_loss_masked(x, y, wav_len: torch.Tensor, n_fft=512, hop=128):
    X = _stft_mag_unfold(x, n_fft=n_fft, hop=hop)  # (B, Freq, frames)
    Y = _stft_mag_unfold(y, n_fft=n_fft, hop=hop)
    if X.numel() == 0:
        return x.new_tensor(0.0)

    B, Freq, frames = X.shape
    frame_idx = torch.arange(frames, device=x.device).unsqueeze(0)  # (1,frames)
    frame_valid = (frame_idx * hop) < wav_len.to(device=x.device).unsqueeze(1)  # (B,frames)
    frame_valid_3 = frame_valid.unsqueeze(1).expand(B, Freq, frames)

    diff = (X - Y).abs() * frame_valid_3.to(dtype=X.dtype)
    denom = frame_valid_3.sum().clamp_min(1).to(dtype=diff.dtype)
    return diff.sum() / denom

def run_train_codec(loader, steps=4000, grad_clip=1.0,
                    w_rec=1.0, w_q=0.4, w_usage=0.2, w_tv=0.1, w_stft=0.6):
    cleanup()
    codec.train()
    it = iter(loader)
    pbar = tqdm(range(steps), desc="train codec", leave=True, position=0)
    ema = None

    for s in pbar:
        try:
            b = next(it)
        except StopIteration:
            it = iter(loader)
            b = next(it)

        b = move_batch(b, device)
        wav = b["wav"]
        has_audio = b["has_audio"]
        if has_audio.sum() == 0:
            continue
        wav = wav[has_audio]
        wav_len = b["wav_len"][has_audio]

        opt.zero_grad(set_to_none=True)
        wav_hat, codes, qloss, dist = codec(wav)

        T = min(wav_hat.size(1), wav.size(1))
        wav_hat = wav_hat[:, :T]
        wav = wav[:, :T]
        wav_len = wav_len.clamp_max(T)

        wav_mask = wav_mask_from_wav_len(wav_len, T, device=wav.device)

        rec = masked_l1(wav_hat, wav, wav_mask)
        tv = masked_tv_diff(wav_hat, wav, wav_mask)

        usage_soft = soft_usage_loss_from_dist(dist, temp=0.5)

        freq = stft_mag_loss_masked(wav_hat, wav, wav_len, n_fft=512, hop=codec.hop)
        diff = masked_hf_diff_loss(wav_hat, wav, wav_mask)

        loss = w_rec*rec + w_tv*tv + w_q*qloss + w_usage*usage_soft + w_stft*freq + w_stft*diff

        loss.backward()
        if grad_clip:
            nn.utils.clip_grad_norm_(codec.parameters(), grad_clip)
        opt.step()

        val = float(loss.detach())
        ema = val if ema is None else 0.98 * ema + 0.02 * val

        H, Eff = usage_metrics(codes, codec.n_codes)
        Hnorm = H / math.log(codec.n_codes)

        pbar.set_postfix({
            "loss": float(loss.detach()),
            "ema": ema,
            "rec": float(rec.detach()),
            "tv": float(tv.detach()),
            "q": float(qloss.detach()),
            "freq": float(freq.detach()),
            "diff": float(diff.detach()),
            "Hnorm": Hnorm,
            "eff_codes": Eff,
            "usage": float(usage_soft.detach()),
        })

#if NOTEBOOK_TRAIN_AND_SAVE_CODEC:
run_train_codec(loader, steps=20000)

In [ ]:
plot_model_matrices(codec)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_codebooks(codec, max_codes_per_book=None, make_square=True):
    """
    Plots the codebook embeddings for each RVQ stage as image(s).

    - Each codebook is an embedding matrix of shape (n_codes, D).
    - We show it as an image: rows=codes, cols=embedding dims.

    Args:
        codec: your AudioCodec
        max_codes_per_book: int or None. If set, only plot the first N codes (rows) per book.
        make_square: if True, try to make the overall figure roughly square by choosing rows/cols.
    """
    codec.eval()
    K = codec.K
    n_codes = codec.n_codes

    # Collect codebook weights (K, n_codes, D)
    books = []
    for k, vq in enumerate(codec.rvq.vqs):
        w = vq.codebook.weight.detach().cpu().numpy()  # (n_codes, D)
        if max_codes_per_book is not None:
            w = w[: int(max_codes_per_book)]
        books.append(w)

    # Decide subplot grid
    n = len(books)
    if make_square:
        ncols = int(math.ceil(math.sqrt(n)))
        nrows = int(math.ceil(n / ncols))
    else:
        ncols = n
        nrows = 1

    # Figure size: scale with number of subplots + matrix aspect
    # (kept moderate so it doesn't explode for large D)
    fig_w = 4.0 * ncols
    fig_h = 3.0 * nrows
    fig, axs = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)

    for idx, w in enumerate(books):
        r = idx // ncols
        c = idx % ncols
        ax = axs[r][c]
        ax.imshow(w, aspect="auto", interpolation="nearest")
        ax.set_title(f"Codebook k={idx}  (rows={w.shape[0]}, D={w.shape[1]})")
        ax.set_xlabel("embedding dim")
        ax.set_ylabel("code index")

    # Hide unused axes
    for idx in range(n, nrows * ncols):
        r = idx // ncols
        c = idx % ncols
        axs[r][c].axis("off")

    suptitle = f"RVQ Codebooks (K={K}, n_codes={n_codes}, D={codec.D})"
    if max_codes_per_book is not None:
        suptitle += f"  |  showing first {max_codes_per_book} codes/book"
    fig.suptitle(suptitle, y=1.02)

    plt.tight_layout()
    plt.show()

# Examples:
# plot_codebooks(codec)                          # all codebooks, all codes
# plot_codebooks(codec, max_codes_per_book=128)  # cap rows to 128 for readability
plot_codebooks(codec, max_codes_per_book=256, make_square=True)

In [ ]:
from IPython.display import Audio, display
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

def _save_wav(path, audio_1d, sr):
    """
    Save a 1D float waveform to .wav on disk.
    Tries soundfile first (keeps float PCM), falls back to scipy (int16).
    """
    audio_1d = np.asarray(audio_1d, dtype=np.float32)

    # Ensure directory exists
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    # Try: soundfile (best for float PCM)
    try:
        import soundfile as sf
        sf.write(path, audio_1d, sr, subtype="PCM_16")  # or "FLOAT"
        return
    except Exception:
        pass

    # Fallback: scipy -> int16
    try:
        from scipy.io.wavfile import write as wav_write
        audio_clipped = np.clip(audio_1d, -1.0, 1.0)
        audio_i16 = (audio_clipped * 32767.0).astype(np.int16)
        wav_write(path, sr, audio_i16)
        return
    except Exception as e:
        raise RuntimeError(
            f"Failed to save wav (need `soundfile` or `scipy`). Last error: {e}"
        )

@torch.no_grad()
def preview_codec_from_loader(codec, loader, device=device, out_dir=".", prefix="sample"):
    codec.eval()

    b = next(iter(loader))
    b = move_batch(b, device)

    has_audio = b["has_audio"]
    if has_audio.sum() == 0:
        print("No audio in batch.")
        return

    i = int(has_audio.nonzero(as_tuple=False)[0].item())

    wav = b["wav"][i]
    wav_len = int(b["wav_len"][i].item())
    sr = int(b.get("audio_sr", SAMPLE_RATE))

    wav = wav[:wav_len].unsqueeze(0)  # (1,T)

    wav_hat, codes, _, _ = codec(wav)

    T = min(wav_hat.size(1), wav.size(1))
    wav_hat = wav_hat[:, :T]
    wav = wav[:, :T]

    wav = wav.squeeze(0).detach().cpu()
    wav_hat = wav_hat.squeeze(0).detach().cpu()

    wav_np = wav.numpy()
    wav_hat_np = wav_hat.numpy()

    # --- Save to disk ---
    orig_path = os.path.join(out_dir, f"{prefix}_orig.wav")
    recon_path = os.path.join(out_dir, f"{prefix}_recon.wav")
    #_save_wav(orig_path, wav_np, sr)
    #_save_wav(recon_path, wav_hat_np, sr)
    #print(f"Saved:\n  {orig_path}\n  {recon_path}")

    # --- Audio preview in notebook ---
    print("ORIGINAL:")
    display(Audio(wav_np, rate=sr))
    print("RECONSTRUCTED:")
    display(Audio(wav_hat_np, rate=sr))

    # --- Plot original vs reconstructed, and abs difference below ---
    diff = np.abs(wav_np - wav_hat_np)
    t = np.arange(T) / float(sr)

    fig, (ax0, ax1) = plt.subplots(
        2, 1, sharex=True, figsize=(12, 5),
        gridspec_kw={"height_ratios": [2, 1]}
    )

    ax0.plot(t, wav_np, label="original", linewidth=1)
    ax0.plot(t, wav_hat_np, label="reconstructed", linewidth=1, alpha=0.8)
    ax0.set_title("Waveforms (Original vs Reconstructed)")
    ax0.set_ylabel("Amplitude")
    ax0.legend(loc="upper right")

    ax1.plot(t, diff, linewidth=1)
    ax1.set_title("Absolute Difference |orig - recon|")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Abs diff")

    plt.tight_layout()
    plt.show()

# usage:
preview_codec_from_loader(codec, loader, device=device, out_dir=".", prefix="codec_preview")

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def show_chunking_and_tokens(codec, loader, device=device, sr_default=SAMPLE_RATE, n_tokens_to_show=200):
    codec.eval()

    b = next(iter(loader))
    b = move_batch(b, device)

    has_audio = b["has_audio"]
    if has_audio.sum() == 0:
        print("No audio in batch.")
        return

    i = int(has_audio.nonzero(as_tuple=False)[0].item())

    wav = b["wav"][i]
    wav_len = int(b["wav_len"][i].item())
    sr = int(b.get("audio_sr", sr_default))

    wav = wav[:wav_len].unsqueeze(0)  # (1,T)
    T = wav.shape[1]

    hop = int(codec.hop)
    token_sec = hop / sr
    tokens_per_sec = sr / hop

    # --- Encode to get latent + codes ---
    z = codec.enc(wav)                 # (1,D,Tf)
    z = torch.tanh(z)
    z_q, codes, qloss, dist = codec.rvq(z)   # codes: (1,K,Tf)
    Tf = z.shape[-1]
    K = codes.shape[1]

    wav_np = wav.squeeze(0).detach().cpu().numpy()
    codes_np = codes.squeeze(0).detach().cpu().numpy()   # (K,Tf)

    # --- Build hop-sized "chunks" directly on the waveform (simple view) ---
    # Note: encoder padding means token receptive fields overlap; this chunking is just
    # the intuitive "hop spacing" visualization.
    n_chunks = math.ceil(T / hop)
    pad = n_chunks * hop - T
    wav_pad = np.pad(wav_np, (0, pad), mode="constant")
    chunks = wav_pad.reshape(n_chunks, hop)   # (n_chunks, hop)

    # per-chunk RMS energy (nice compact proxy)
    rms = np.sqrt(np.mean(chunks**2, axis=1))  # (n_chunks,)

    # Time axes
    t = np.arange(T) / sr
    chunk_centers = (np.arange(n_chunks) * hop + hop/2) / sr
    token_centers = (np.arange(Tf) * hop + hop/2) / sr  # approximate alignment

    # Optionally zoom to first N tokens to keep plots tight
    Tf_show = min(Tf, n_tokens_to_show)
    t_max = (Tf_show * hop) / sr
    t_mask = t <= t_max

    # --- Plot ---
    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(3 + K, 1, height_ratios=[2.0, 1.0, 0.6] + [0.6]*K, hspace=0.2)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
    ax2 = fig.add_subplot(gs[2, 0], sharex=ax0)
    axs_codes = [fig.add_subplot(gs[3 + k, 0], sharex=ax0) for k in range(K)]

    # 1) Waveform with hop boundaries
    ax0.plot(t[t_mask], wav_np[t_mask], linewidth=1)
    ax0.set_title(
        f"Waveform with hop boundaries  |  hop={hop} samples = {1000*token_sec:.2f} ms  |  tokens/sec={tokens_per_sec:.2f}  |  Tf={Tf}"
    )
    ax0.set_ylabel("amp")

    # vertical lines at hop boundaries (up to Tf_show)
    for n in range(Tf_show + 1):
        x = (n * hop) / sr
        ax0.axvline(x, linewidth=0.5, alpha=0.3)

    # 2) Chunk RMS energy (hop-sized)
    # align rms to chunk centers
    mask_chunks = chunk_centers <= t_max
    ax1.plot(chunk_centers[mask_chunks], rms[mask_chunks], linewidth=1)
    ax1.set_title("Per-hop chunk RMS energy (computed directly on waveform)")
    ax1.set_ylabel("RMS")

    # 3) Token index timeline (0..Tf_show-1) as a sanity-check ruler
    ax2.plot(token_centers[:Tf_show], np.arange(Tf_show), linewidth=1)
    ax2.set_title("Token centers (approx) vs token index")
    ax2.set_ylabel("token #")

    # 4) Code indices over time for each residual VQ codebook
    for k, axk in enumerate(axs_codes):
        axk.step(token_centers[:Tf_show], codes_np[k, :Tf_show], where="mid", linewidth=1)
        axk.set_ylabel(f"code{k}")
        if k == 0:
            axk.set_title("RVQ code indices per token (one row per codebook)")

    axs_codes[-1].set_xlabel("time (s)")
    plt.setp(ax0.get_xticklabels(), visible=False)
    plt.setp(ax1.get_xticklabels(), visible=False)
    plt.setp(ax2.get_xticklabels(), visible=False)
    for axk in axs_codes[:-1]:
        plt.setp(axk.get_xticklabels(), visible=False)

    plt.show()

    # Print a compact summary
    print(f"T={T} samples ({T/sr:.3f} s), sr={sr}")
    print(f"hop={hop} samples => {1000*hop/sr:.2f} ms per token step, tokens/sec={sr/hop:.2f}")
    print(f"Encoder output Tf={Tf} tokens (latent frames). Showing first {Tf_show} tokens.")
    print(f"codes shape: {tuple(codes.shape)}  (B,K,Tf)")

# Usage:
show_chunking_and_tokens(codec, loader, device=device, sr_default=SAMPLE_RATE, n_tokens_to_show=200)

In [ ]:
torch.save(codec.state_dict(), "../artifact/audio.codec.lg.pt")  # save